##### Support Request Agent Stream

Stream support requests through the support agent endpoint and persist reports.

Uses fake-it-till-up pattern: deterministic responses during backfill and when
the agent endpoint is not yet ready, real inference with capping once available.

In [ ]:
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().getOrElse(None)
DATABRICKS_HOST = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().getOrElse(None)

CATALOG = dbutils.widgets.get("CATALOG")
SUPPORT_AGENT_ENDPOINT_NAME = dbutils.widgets.get("SUPPORT_AGENT_ENDPOINT_NAME")

In [ ]:
%pip install openai

In [ ]:
import json
import random
import os

from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf
from pyspark.sql.window import Window
from openai import OpenAI

CHECKPOINT_PATH = f"/Volumes/{CATALOG}/support/checkpoints/support_request_agent_stream"
MAX_INFERENCES_PER_BATCH = 50


def is_first_run():
    """Check if checkpoint exists (indicates this is NOT the first run)"""
    return not os.path.exists(CHECKPOINT_PATH) or len(os.listdir(CHECKPOINT_PATH)) == 0


# ── Deterministic fallback responses ──────────────────────────────────────────
# These mirror the policy resolution logic from the agent, providing realistic
# structured responses when the endpoint is unavailable or during backfill.

fake_responses = [
    {
        "support_request_id": "", "user_id": "", "order_id": "",
        "credit_recommendation": {"amount_usd": 3.0, "reason": "Goodwill credit for minor service issue."},
        "refund_recommendation": {"amount_usd": 0.0, "reason": "No verified severe failure."},
        "draft_response": "Thanks for contacting us, and I am really sorry about this experience. We are offering a $3.00 credit on this case.",
        "past_interactions_summary": "No prior interactions found.",
        "order_details_summary": "Standard delivery order.",
        "decision_confidence": "medium",
        "escalation_flag": False,
    },
    {
        "support_request_id": "", "user_id": "", "order_id": "",
        "credit_recommendation": {"amount_usd": 5.0, "reason": "Moderate service issue with customer impact."},
        "refund_recommendation": {"amount_usd": 10.0, "reason": "Moderate service issue with customer impact."},
        "draft_response": "Thanks for contacting us, and I am really sorry about this experience. We are offering a $10.00 refund and a $5.00 credit on this case.",
        "past_interactions_summary": "No prior interactions found.",
        "order_details_summary": "Delivery with reported delay.",
        "decision_confidence": "high",
        "escalation_flag": False,
    },
    {
        "support_request_id": "", "user_id": "", "order_id": "",
        "credit_recommendation": {"amount_usd": 2.0, "reason": "Minor issue or low-confidence compensable case."},
        "refund_recommendation": {"amount_usd": 0.0, "reason": "No severe failure identified."},
        "draft_response": "Thanks for contacting us, and I am really sorry about this experience. We are offering a $2.00 credit on this case.",
        "past_interactions_summary": "No prior interactions found.",
        "order_details_summary": "Standard delivery order.",
        "decision_confidence": "medium",
        "escalation_flag": False,
    },
    {
        "support_request_id": "", "user_id": "", "order_id": "",
        "credit_recommendation": {"amount_usd": 5.0, "reason": "Repeated or high-risk service failure."},
        "refund_recommendation": {"amount_usd": 18.0, "reason": "Repeated or high-risk service failure."},
        "draft_response": "I see this is at least the 2nd time you've had to contact us for similar issues, and that is not acceptable. We are offering a $18.00 refund and a $5.00 credit on this case.",
        "past_interactions_summary": "Multiple prior contacts for similar issues.",
        "order_details_summary": "Repeat issue with delivery.",
        "decision_confidence": "high",
        "escalation_flag": False,
    },
    {
        "support_request_id": "", "user_id": "", "order_id": "",
        "credit_recommendation": {"amount_usd": 3.0, "reason": "Abusive/spammy pattern detected; restricted compensation with escalation."},
        "refund_recommendation": {"amount_usd": 0.0, "reason": "Abusive/spammy pattern detected."},
        "draft_response": "Thanks for contacting us. We are escalating this case for manual review and will follow up quickly.",
        "past_interactions_summary": "Pattern of escalation detected.",
        "order_details_summary": "Order under review.",
        "decision_confidence": "medium",
        "escalation_flag": True,
    },
]


def get_fake_response(support_request_id: str, user_id: str, order_id: str) -> str:
    """Return a deterministic fake response with IDs filled in."""
    resp = random.choice(fake_responses).copy()
    resp["support_request_id"] = support_request_id
    resp["user_id"] = user_id
    resp["order_id"] = order_id
    return json.dumps(resp)


def call_support_agent(payload_json: str) -> str:
    """Call the support agent endpoint with the case payload."""
    payload = json.loads(payload_json)
    support_request_id = payload.get("support_request_id", "")
    user_id = payload.get("user_id", "")
    order_id = payload.get("order_id", "")

    client = OpenAI(
        api_key=DATABRICKS_TOKEN,
        base_url=f"{DATABRICKS_HOST}/serving-endpoints",
    )

    for _ in range(3):
        try:
            response_obj = client.responses.create(
                model=SUPPORT_AGENT_ENDPOINT_NAME,
                input=[{"role": "user", "content": payload_json}],
            )
            response = response_obj.output[-1].content[0].text
            json.loads(response)  # validate JSON
            return response
        except Exception:
            continue

    # All retries failed — return fake
    return get_fake_response(support_request_id, user_id, order_id)


call_support_agent_udf = udf(call_support_agent, StringType())
get_fake_response_udf = udf(get_fake_response, StringType())

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.support")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.support.checkpoints")

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.support.support_agent_reports (
  support_request_id STRING,
  user_id STRING,
  order_id STRING,
  request_text STRING,
  ts TIMESTAMP,
  agent_response STRING
)
""")

In [ ]:
# Enable CDC for Lakebase sync
table_name = f"{CATALOG}.support.support_agent_reports"

try:
    props = spark.sql(f"SHOW TBLPROPERTIES {table_name}").collect()
    cdc_enabled = any(row.key == "delta.enableChangeDataFeed" and row.value == "true" for row in props)
except Exception:
    cdc_enabled = False

if not cdc_enabled:
    print(f"Enabling CDC on {table_name}")
    spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
else:
    print(f"CDC already enabled on {table_name}, skipping")

In [ ]:
def process_batch(batch_df, batch_id):
    """Process each micro-batch with inference capping.

    - First run (no checkpoint): ALL rows get fake responses (fast backfill)
    - Subsequent runs: First N rows get real inference, rest get fake
    """
    if batch_df.isEmpty():
        return

    first_run = is_first_run()
    row_count = batch_df.count()

    print(f"Processing batch {batch_id}: {row_count} rows, first_run={first_run}")

    # Build payload JSON from row columns for the agent
    with_payload = batch_df.withColumn(
        "payload_json",
        F.to_json(
            F.struct(
                F.col("support_request_id"),
                F.col("user_id"),
                F.col("order_id"),
                F.col("request_type"),
                F.col("edge_case_type"),
                F.col("request_text"),
            )
        ),
    )

    if first_run:
        # First run: ALL rows get fake responses (fast backfill)
        print(f"  -> First run detected, using fake responses for all {row_count} rows")
        result_df = with_payload.select(
            F.col("support_request_id"),
            F.col("user_id"),
            F.col("order_id"),
            F.col("request_text"),
            F.current_timestamp().alias("ts"),
            get_fake_response_udf(
                F.col("support_request_id"), F.col("user_id"), F.col("order_id")
            ).alias("agent_response"),
        )
    else:
        # Subsequent runs: first N rows get real inference, rest get fake
        windowed = with_payload.withColumn(
            "row_num",
            F.row_number().over(Window.orderBy(F.col("ts"), F.col("support_request_id"))),
        )

        real_count = min(row_count, MAX_INFERENCES_PER_BATCH)
        fake_count = max(0, row_count - MAX_INFERENCES_PER_BATCH)
        print(f"  -> Real inference: {real_count} rows, fake: {fake_count} rows")

        real_inference_df = windowed.filter(f"row_num <= {MAX_INFERENCES_PER_BATCH}").select(
            F.col("support_request_id"),
            F.col("user_id"),
            F.col("order_id"),
            F.col("request_text"),
            F.current_timestamp().alias("ts"),
            call_support_agent_udf(F.col("payload_json")).alias("agent_response"),
        )

        fake_response_df = windowed.filter(f"row_num > {MAX_INFERENCES_PER_BATCH}").select(
            F.col("support_request_id"),
            F.col("user_id"),
            F.col("order_id"),
            F.col("request_text"),
            F.current_timestamp().alias("ts"),
            get_fake_response_udf(
                F.col("support_request_id"), F.col("user_id"), F.col("order_id")
            ).alias("agent_response"),
        )

        result_df = real_inference_df.union(fake_response_df)

    result_df.write.mode("append").saveAsTable(f"{CATALOG}.support.support_agent_reports")

In [ ]:
# Read stream from raw support requests
support_requests = spark.readStream.table(f"{CATALOG}.support.raw_support_requests")

support_requests.writeStream \
    .foreachBatch(process_batch) \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .trigger(availableNow=True) \
    .start() \
    .awaitTermination()